# Marker Repo - annotation

In this notebook, clustered h5ad files can be annotated using the marker repo.

## Loading packages

In [ ]:
import markerrepo.marker_repo as mr
import markerrepo.wrappers as wrap
import markerrepo.annotation as annot
import scanpy as sc

%load_ext autoreload
%autoreload 2

## Settings

Specify path of the cloned repository, the h5ad file which is going to be annotated as well as the organism.

In [ ]:
repo_path = "/mnt/workspace/mkessle/projects/annotate_by_marker_and_features"
h5ad_path = "/mnt/agnerds/user/micha.kessler/zebrafish.h5ad"
organism = mr.select(key="organism")

Load anndata

In [ ]:
adata = sc.read_h5ad(h5ad_path)

## Prepare annotation

Pick the clustring column you want to annotate.

In [ ]:
column = mr.select(whitelist=list(adata.obs.columns), heading="clustering column")

Rank genes

In [ ]:
print(f'Ranking genes groups for clusters using obs column {column}')
sc.tl.rank_genes_groups(adata, groupby=f'{column}', use_raw=False, key_added=f'rank_genes_groups_{column}')

Create suitable marker list(s)

In [ ]:
# TODO

Use homology to create marker list(s)

In [ ]:
marker_lists = wrap.transfer_markers(target_org=organism, source_df=None, repo_path=repo_path, target_counts=1, 
                      weight_markers=True, export_suffix="annotation", ui=True)

## Annotation

In [ ]:
for marker_list in marker_lists:
    name = marker_list.split("/")[-1]
    annotation_dir = f"./annotation/{name}"
    
    # Annotate
    annot.annot_ct(adata=adata, genes_adata=adata, output_path=annotation_dir, db_path=marker_list,
                   cluster_column=f"{column}", rank_genes_column=f"rank_genes_groups_{column}", 
                   ct_column=f"cell_types_{name}", tissue="all", inplace=True)
    
    # Plot annotation
    sc.pl.umap(adata, color=[f'cell_types_{name}', f'{column}'], wspace=0.5)

    # Show scores and alternate cell types of eacht cluster
    print(f"Tables of cell type annotation with clustering {column}")
    annot.show_tables(annotation_dir=annotation_dir, n=5, clustering_column=column)